___
# <center>Causalidade e viés de seleção em dados de processos</center>
___

## Clínica BTG · 19/10/2026

**Objetivos de aprendizagem.** Ao final, você deve ser capaz de:

 * classificar uma pergunta do projeto como de previsão, de intervenção ou
   contrafactual;
 * reconhecer quando uma terceira variável explica uma associação na base, e
   ajustar por ela comparando dentro de grupos parecidos;
 * identificar viés de seleção, inclusive na escolha de quais disputas chegam a
   julgamento;
 * dizer o que a conta de custo esperado responde e o que só um experimento
   responde.

___
<div id="indice"></div>

## Índice

- [Parte 1: juros mais altos, menos pagamento? (variável de confusão)](#confusao)
- [Parte 2: disputas que chegam a julgamento (viés de seleção)](#selecao)
- [Parte 3: custo esperado e a decisão de acordo](#decisao)
- [Exercício](#exercicio)

> **Sobre a base.** Os dados são do projeto da clínica com o BTG e estão cobertos
> por sigilo bancário e pelo termo de parceria. O arquivo `btg_aulas.csv` é uma
> amostra de 800 processos da base que vocês já receberam, com colunas
> estruturadas a partir dos PDFs. O arquivo `btg_juros.csv` tem os 28.896
> processos da planilha com taxa de juros válida, só com ano de ajuizamento,
> tribunal, taxa e pagamento. **Nenhum dos dois pode ser compartilhado** fora da
> clínica, nem subido para o GitHub, nem colado em ferramentas externas.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_aulas.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
processos = pd.read_csv("btg_aulas.csv")
processos.shape

[Volta ao Índice](#indice)

___
<div id="confusao"></div>

# Parte 1: juros mais altos, menos pagamento?

As ações da carteira são revisionais e, em geral, alegam juros abusivos. Seria
de esperar que contratos com juros mais altos terminassem em pagamento com mais
frequência. Vamos olhar a planilha completa.

In [ ]:
import numpy as np

juros = pd.read_csv("btg_juros.csv")
print(f"{len(juros)} processos")

juros["faixa"] = pd.cut(juros["taxa_juros"], [0, 2, 2.5, 3, 99],
                        labels=["até 2%", "2% a 2,5%", "2,5% a 3%", "acima de 3%"])
juros.groupby("faixa", observed=True)["pagou"].agg(["size", "mean"]).round(3)

O contrário: de 14,0% nos juros até 2% para 9,6% nos acima de 3%. A leitura
apressada seria **"juros altos protegem o banco"**.

Antes de recomendar, pergunte: **o que mais difere entre contratos de juros
altos e baixos?** Uma candidata é a época. Os juros subiram a partir de 2022, e
processos recentes tiveram menos tempo para terminar em acordo ou condenação.

In [ ]:
juros["periodo"] = np.where(juros["ano_ajuizamento"] <= 2021, "até 2021", "2022 em diante")

# juros médio e taxa de pagamento em cada período
juros.groupby("periodo")[["____", "pagou"]].mean().round(3)

Os processos de 2022 em diante têm juros mais altos (2,8% contra 2,2% ao mês) e
pagam bem menos (6,7% contra 17,5%). O período afeta as duas coisas ao mesmo
tempo: é candidato a **variável de confusão**.

**Ajustar** pelo período é comparar faixas de juros **dentro** de cada período.

In [ ]:
# taxa de pagamento por faixa de juros, dentro de cada período
tabela = juros.pivot_table(index="____", columns="faixa", values="pagou",
                           aggfunc="mean", observed=True)
tabela.round(3)

Dentro de **até 2021**, a taxa de pagamento **sobe** com os juros (de 16,8% para
23,4%). Dentro de **2022 em diante**, fica entre 6% e 8%, e é maior na faixa
mais alta. A queda geral vinha da mistura de períodos: os juros altos se
concentram nos processos novos, que pagam menos com qualquer taxa. É o
**paradoxo de Simpson**.

Duas lições para o projeto:

1. **Ajustar é comparar dentro de grupos parecidos.** Aqui, processos do mesmo
   período. A escolha do grupo vem do raciocínio sobre o mecanismo (o que afeta
   as duas variáveis), e não de testar variáveis até o resultado mudar.
2. **Controlar só resolve o que foi medido.** Tribunal, tipo de veículo e
   escritório do autor também podem afetar juros e pagamento. O que não está
   na base continua podendo distorcer a comparação.

[Volta ao Índice](#indice)

___
<div id="selecao"></div>

# Parte 2: disputas que chegam a julgamento

A hipótese de Priest e Klein (1984), de forma simplificada: quando as duas
partes conseguem prever o resultado, sai mais barato fazer acordo. Só vão a
julgamento as disputas **perto do limiar** de decisão, em que o resultado é
incerto.

Vamos simular. Cada disputa tem uma **força do caso do autor** entre 0 e 1. O
juiz dá vitória ao autor se a força, com um pouco de ruído, passa do
**limiar** do padrão jurídico. As partes enxergam a força com pouco erro e só
litigam quando ela está a menos de 0,1 do limiar.

In [ ]:
rng = np.random.default_rng(1984)

def simula(limiar, n=4000, faixa=0.1):
    forca = rng.uniform(0, 1, n)
    vence = forca + rng.normal(0, 0.06, n) > limiar
    julgada = np.abs(forca + rng.normal(0, 0.03, n) - limiar) < faixa
    return pd.DataFrame({"forca": forca, "autor_vence": vence, "julgada": julgada})

lei_pro_autor = simula(limiar=0.3)
lei_pro_reu = simula(limiar=0.7)

print(f"se todas fossem julgadas: autor venceria {lei_pro_autor['autor_vence'].mean():.0%} "
      f"(lei pró-autor) e {lei_pro_reu['autor_vence'].mean():.0%} (lei pró-réu)")

In [ ]:
# a taxa de vitória do autor só entre as disputas que foram a julgamento
for nome, sim in [("lei pró-autor", lei_pro_autor), ("lei pró-réu", lei_pro_reu)]:
    julgadas = sim[sim["____"]]
    print(f"{nome}: {len(julgadas)} julgadas, autor vence {julgadas['autor_vence'].mean():.0%}")

Entre as julgadas, o autor vence **perto de metade** nos dois cenários, embora a
lei seja muito diferente. A taxa de vitória nas sentenças descreve as disputas
que **sobraram** depois dos acordos, e não a força da lei nem a chance de um
processo qualquer.

**Na base do BTG.** A planilha só tem conflitos que viraram processo, e o que
aconteceu com eles depende da política atual de acordos do banco. Antes de
comparar grupos, descreva como os processos chegaram à base e quem ficou de
fora.

[Volta ao Índice](#indice)

___
<div id="decisao"></div>

# Parte 3: custo esperado e a decisão de acordo

Uma conta simples para comparar com uma proposta de acordo:

$$\text{custo esperado} = P(\text{pagar}) \times \text{valor médio pago}$$

In [ ]:
valor_medio = processos.loc[processos["pagou"], "valor_pago"].mean()
print(f"valor médio pago, entre os que pagaram: R$ {valor_medio:,.0f}")

trib = processos.groupby("tribunal")["pagou"].agg(n="size", taxa="mean")
trib["custo_esperado"] = trib["taxa"] * valor_medio
trib.loc[["TJMG", "TJRS", "TJSP", "TJRJ"]].round(3)

Com uma proposta de acordo de **R$ 500** (valor hipotético, só para a conta), o
banco faria acordo nos processos do TJMG (custo esperado perto de R$ 650) e
litigaria nos demais.

**O que essa conta não responde.** O `valor_pago` da planilha **já inclui
acordos**, e os processos que foram a acordo foram escolhidos. A taxa estimada é
o que aconteceu com a política que o banco já segue, e não o que aconteceria com
outra. Saber se **oferecer** acordo reduz o custo é uma pergunta de
**intervenção**: pede processos parecidos com e sem oferta. O jeito mais limpo
de garantir que sejam parecidos é **sortear** quais recebem a oferta, numa parte
da carteira.

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# Exercício

1. Na simulação da parte 2, refaça a conta com limiar 0,5 e com limiar 0,9. A
   taxa de vitória do autor entre as julgadas muda muito?
2. Agora suponha que as partes errem muito mais na previsão e litiguem casos
   mais claros: use `simula(0.3, faixa=0.5)`. O que acontece com o número de
   julgadas e com a taxa de vitória entre elas? Por quê?
3. Na parte 1, troque o período pelo tribunal, em dois grupos: TJSP e demais
   (`np.where(juros["tribunal"] == "TJSP", "TJSP", "demais")`). A queda do
   pagamento com os juros some? Depois separe por tribunal **e** período ao
   mesmo tempo (`index=["sp", "periodo"]`). O que muda?

In [ ]:
# item 1

In [ ]:
# item 2

In [ ]:
# item 3

[Volta ao Índice](#indice)